In [ ]:
#| default_exp context

# Context search

> Fast, structured notebook context for AI coding agents: project maps, chapter bundles, symbol stories, and search.

Instead of reading raw notebook JSON or juggling twenty query flags, agents answer three concrete questions: where code lives (`nb_map`), what one chapter or symbol means (`chapter_context`, `symbol_context`), and where a pattern appears (`search_nb`).

In [ ]:
#| export
import ast, difflib, re
from dataclasses import dataclass, field
from pathlib import Path

from fastcore.nbio import read_nb

from nbskill.core import (cell_defs, cell_directives, cell_source, chapters, exported_py_path,
    heading_info, is_exported, is_hidden, notebook_paths, parse_cell_ast, project_root, source_hash)
from nbskill.state import notebook_state

In [ ]:
from tempfile import TemporaryDirectory
from fastcore.nbio import mk_cell, new_nb, write_nb
from fastcore.test import test_eq, test_fail

## Project and notebook map

`nb_map` gives an agent an immediate bird's-eye view of a project or single notebook: title, subtitle, generated module path, hidden-cell ratio, and every H2 chapter with its exported public symbols.

In [ ]:
#| export
@dataclass
class ChapterSummary:
    "Compact metadata for one H2 chapter in a notebook map."
    title: str
    start: int
    end: int
    defs: list[str] = field(default_factory=list)

@dataclass
class NotebookSummary:
    "Map entry for one notebook."
    path: str
    title: str
    subtitle: str
    exported_py: str | None
    cell_count: int
    code_count: int
    hidden_count: int
    chapters: list[ChapterSummary] = field(default_factory=list)

    @property
    def hidden_pct(self) -> int: return round(100 * self.hidden_count / self.code_count) if self.code_count else 0

@dataclass
class ProjectMap:
    "Structured outline of one or more notebooks."
    root: str
    notebooks: list[NotebookSummary] = field(default_factory=list)

    def __str__(self) -> str:
        lines = []
        for nb in self.notebooks:
            exp = f" -> {nb.exported_py}" if nb.exported_py else ""
            lines.append(f"{nb.path} — {nb.title} ({nb.cell_count} cells, {nb.hidden_pct}% hidden{exp})")
            if nb.subtitle: lines.append(f"  > {nb.subtitle}")
            for ch in nb.chapters:
                syms = f": {', '.join(ch.defs)}" if ch.defs else ""
                lines.append(f"  - {ch.title} [{ch.start}..{ch.end}]{syms}")
        return "\n".join(lines) or f"No notebooks under {self.root}"

    def _repr_markdown_(self) -> str: return f"```text\n{self}\n```"

Title and subtitle follow nbdev's convention: the first H1 Markdown heading and the first `>` blockquote line.

In [ ]:
#| export
def _title_and_subtitle(nb) -> tuple[str, str]:
    title, sub = "", ""
    for c in nb.cells[:6]:
        if c.get("cell_type") != "markdown": continue
        for line in cell_source(c).splitlines():
            s = line.strip()
            if not title and s.startswith("# "): title = s[2:].strip()
            elif not sub and s.startswith(">"): sub = s[1:].strip()
    return title or "(untitled)", sub

def _rel_label(p: Path, root: Path) -> str:
    try: return p.resolve().relative_to(root.resolve()).as_posix()
    except ValueError: return str(p)

def nb_map(path: str | Path = ".") -> ProjectMap:
    "Map notebooks, H2 chapters, exported symbols, and hidden-cell ratios under `path`."
    p = Path(path).expanduser()
    root = project_root(p)
    entries = []
    for nb_path in notebook_paths(p):
        nb = read_nb(nb_path)
        title, sub = _title_and_subtitle(nb)
        code_cells = [c for c in nb.cells if c.get("cell_type") == "code"]
        hidden = sum(is_hidden(c) for c in code_cells)
        exp_py = exported_py_path(nb_path, nb)
        exp_label = _rel_label(exp_py, root) if exp_py else None
        chaps = [ChapterSummary(ch.title, ch.start, ch.end, ch.defs) for ch in chapters(nb) if ch.title != "Notebook head" or ch.defs]
        entries.append(NotebookSummary(_rel_label(nb_path, root), title, sub, exp_label, len(nb.cells), len(code_cells), hidden, chaps))
    return ProjectMap(str(p), entries)

Running `nb_map` on a small notebook reports its title, chapter symbols, and hidden percentage in one call.

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp) / "00_demo.ipynb"
    write_nb(new_nb([mk_cell("# Demo", "markdown"), mk_cell("> Tiny guide.", "markdown"),
        mk_cell("## Math", "markdown"), mk_cell("#| export\ndef add(a, b): return a + b"), mk_cell("#| hide\nassert add(1, 2) == 3")]), p)
    m = nb_map(p)
    test_eq((m.notebooks[0].title, m.notebooks[0].subtitle, m.notebooks[0].hidden_pct), ("Demo", "Tiny guide.", 50))
    test_eq(m.notebooks[0].chapters[0].defs, ["add"])
m

## Chapter context

Once `nb_map` identifies the right chapter, `chapter_context` returns every cell of that chapter in reading order with stable cell IDs, directives, source hashes, and execution freshness.

In [ ]:
#| export
@dataclass
class CellView:
    "Compact rendered view of one notebook cell."
    cell_id: str
    cell_idx: int
    cell_type: str
    directives: dict[str, str]
    source: str
    source_hash: str
    state: str = ""
    output: str = ""

@dataclass
class ChapterContext:
    "Complete reading bundle for one H2 chapter."
    path: str
    title: str
    cells: list[CellView]

    def __str__(self) -> str:
        out = [f"# {self.path} :: {self.title}"]
        for c in self.cells:
            tags = [f"#| {k}" if v == "true" else f"#| {k}: {v}" for k, v in c.directives.items()]
            meta = f" [{' '.join(tags)}]" if tags else ""
            st = f" state={c.state}" if c.state else ""
            out.append(f"\n--- {c.cell_id} ({c.cell_type}{st}){meta} ---")
            out.append(c.source)
            if c.output: out.append(f"[output] {c.output}")
        return "\n".join(out)

    def _repr_markdown_(self) -> str: return f"```text\n{self}\n```"

Output previews stay short so large saved tables or logs never flood an agent's context window.

In [ ]:
#| export
def _output_preview(cell, max_chars: int = 240) -> str:
    parts = []
    for o in cell.get("outputs") or []:
        ot = o.get("output_type", "")
        if ot == "stream": parts.append("".join(o.get("text", "")))
        elif ot in ("execute_result", "display_data"):
            data = o.get("data") or {}
            txt = data.get("text/plain") or data.get("text/markdown") or ""
            parts.append("".join(txt) if isinstance(txt, list) else str(txt))
        elif ot == "error": parts.append(f"{o.get('ename', 'Error')}: {o.get('evalue', '')}")
    joined = " | ".join(p.strip() for p in parts if p.strip())
    return joined if len(joined) <= max_chars else joined[:max_chars] + "..."

def _cell_view(cell, idx: int, state_map: dict, include_outputs: bool) -> CellView:
    cid = str(cell.get("id", f"cell-{idx}"))
    src = cell_source(cell)
    st = state_map[cid].state if cid in state_map else ""
    out = _output_preview(cell) if include_outputs else ""
    return CellView(cid, idx, cell.get("cell_type", "code"), cell_directives(cell), src, source_hash(src), st, out)

def chapter_context(
    path: str | Path,  # Notebook path
    chapter: str,  # Exact or case-insensitive substring match for an H2 chapter title
    *,
    include_outputs: bool = False,  # Include truncated saved outputs
) -> ChapterContext:
    "Return one H2 chapter in reading order with cell IDs, hashes, and freshness."
    p = Path(path).expanduser()
    nb = read_nb(p)
    chaps = chapters(nb)
    needle = chapter.strip().lower()
    matched = [ch for ch in chaps if ch.title.lower() == needle] or [ch for ch in chaps if needle in ch.title.lower()]
    if not matched:
        avail = ", ".join(ch.title for ch in chaps)
        raise ValueError(f"Chapter {chapter!r} not found in {p.name}. Available: {avail}")
    chosen = matched[0]
    st_map = notebook_state(p).by_id()
    views = [_cell_view(c, chosen.start + i, st_map, include_outputs) for i, c in enumerate(chosen.cells)]
    return ChapterContext(str(p), chosen.title, views)

Selecting a chapter returns its cells with freshness state; asking for an unknown chapter lists the valid titles.

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp) / "guide.ipynb"
    write_nb(new_nb([mk_cell("## Billing", "markdown"), mk_cell("Explain billing.", "markdown"), mk_cell("total = 10")]), p)
    ch = chapter_context(p, "billing")
    test_eq((ch.title, len(ch.cells), ch.cells[-1].state), ("Billing", 3, "unknown"))
    test_fail(lambda: chapter_context(p, "missing"), contains="Available: Billing")
ch

## Symbol context

Before changing a function, class, or `@patch` method, an agent needs its contract explanation, definition, nearby examples and assertions, static callers across the project, and execution freshness. `symbol_context` assembles all five in one call.

In [ ]:
#| export
@dataclass
class CallerRef:
    "One static call site referencing a symbol."
    path: str
    chapter: str
    cell_id: str
    lineno: int

@dataclass
class SymbolContext:
    "Complete pre-edit bundle for one symbol across its owning chapter and callers."
    symbol: str
    path: str
    chapter: str
    chapter_intro: list[CellView]
    definitions: list[CellView]
    explanations: list[CellView]
    examples: list[CellView]
    distant_examples: list[tuple[str, str]]
    callers: list[CallerRef]

    def __str__(self) -> str:
        lines = [f"# Symbol {self.symbol} in {self.path} :: {self.chapter}"]
        if self.explanations:
            lines.append("\n## Explanations")
            for c in self.explanations: lines.append(f"[{c.cell_id}] {c.source}")
        lines.append("\n## Definition")
        for c in self.definitions: lines.append(f"[{c.cell_id} hash={c.source_hash} state={c.state}]\n{c.source}")
        if self.examples:
            lines.append("\n## Nearby examples & checks")
            for c in self.examples:
                out = f"\n[output] {c.output}" if c.output else ""
                lines.append(f"[{c.cell_id} state={c.state}]\n{c.source}{out}")
        elif self.distant_examples:
            where = ", ".join(f"{ch}#{cid}" for ch, cid in self.distant_examples)
            lines.append(f"\n## Examples\nNone in this chapter (found elsewhere: {where})")
        if self.callers:
            lines.append("\n## Static callers")
            for ref in self.callers[:15]: lines.append(f"- {ref.path}#{ref.cell_id}:{ref.lineno} ({ref.chapter})")
        return "\n".join(lines)

    def _repr_markdown_(self) -> str: return f"```text\n{self}\n```"

Static call discovery walks AST `Call` nodes and checks both bare function names (`foo(...)`) and attribute calls (`obj.foo(...)`).

In [ ]:
#| export
def _call_names_with_lines(cell) -> list[tuple[str, int]]:
    tree, _ = parse_cell_ast(cell)
    if tree is None: return []
    out = []
    for node in ast.walk(tree):
        if not isinstance(node, ast.Call): continue
        fn = node.func
        if isinstance(fn, ast.Name): out.append((fn.id, node.lineno))
        elif isinstance(fn, ast.Attribute): out.append((fn.attr, node.lineno))
    return out

def _matches_symbol(d, query: str) -> bool:
    if "." in query:
        recv, name = query.split(".", 1)
        return d.name == name and d.receiver == recv
    return d.name == query

def _cell_mentions_symbol(cell, name: str) -> bool:
    tree, _ = parse_cell_ast(cell)
    if tree is None: return False
    return any((isinstance(n, ast.Name) and n.id == name) or (isinstance(n, ast.Attribute) and n.attr == name) for n in ast.walk(tree))

`symbol_context` accepts either `symbol` (searched across `scope`) or `path/to/nb.ipynb#symbol`. If no exact match exists, it suggests close symbol names.

In [ ]:
#| export
def _chapter_for_idx(chaps: list, idx: int):
    for ch in chaps:
        if ch.start <= idx < ch.end: return ch
    return chaps[0]

def _find_callers(short_name: str, scope: Path, def_ids: set) -> list[CallerRef]:
    root = project_root(scope)
    refs = []
    for nb_path in notebook_paths(scope):
        nb = read_nb(nb_path)
        chaps = chapters(nb)
        rel = _rel_label(nb_path, root)
        for idx, cell in enumerate(nb.cells):
            cid = str(cell.get("id", f"cell-{idx}"))
            if (str(nb_path.resolve()), cid) in def_ids: continue
            ch_title = _chapter_for_idx(chaps, idx).title
            for cname, lineno in _call_names_with_lines(cell):
                if cname == short_name: refs.append(CallerRef(rel, ch_title, cid, lineno))
    return refs

`symbol_context` resolves a symbol (or `notebook.ipynb#symbol`), gathers nearby explanations and examples in its owning chapter, and attaches static callers and execution freshness.

In [ ]:
#| export
def _collect_symbol_hits(search_path: Path, sym: str):
    all_names, hits = set(), []
    for nb_path in notebook_paths(search_path):
        nb = read_nb(nb_path)
        for idx, cell in enumerate(nb.cells):
            for d in cell_defs(cell):
                all_names.add(f"{d.receiver}.{d.name}" if d.receiver else d.name)
                all_names.add(d.name)
                if _matches_symbol(d, sym): hits.append((nb_path, nb, idx, cell))
    return all_names, hits

def symbol_context(
    target: str,  # Symbol name (e.g. `diff_nb`, `Box.label`) or `notebook.ipynb#symbol`
    scope: str | Path = ".",  # Directory or notebook to search
) -> SymbolContext:
    "Return explanation, definition, nearby examples, callers, and freshness for one symbol."
    if "#" in target:
        nb_part, sym = target.rsplit("#", 1)
        search_path, scope_path = Path(nb_part).expanduser(), Path(scope).expanduser()
    else: sym, search_path, scope_path = target.strip(), Path(scope).expanduser(), Path(scope).expanduser()
    short_name = sym.split(".", 1)[-1]
    all_names, hits = _collect_symbol_hits(search_path, sym)
    if not hits:
        close = difflib.get_close_matches(sym, sorted(all_names), n=5)
        hint = f" Closest: {', '.join(close)}" if close else ""
        raise ValueError(f"Symbol {sym!r} not found under {search_path}.{hint}")
    nb_path, nb, first_idx, _ = hits[0]
    chaps = chapters(nb)
    chap = _chapter_for_idx(chaps, first_idx)
    st_map = notebook_state(nb_path).by_id()
    same_nb = [(i, c) for p, _, i, c in hits if p.resolve() == nb_path.resolve()]
    def_idxs = {i for i, _ in same_nb}
    defs = [_cell_view(c, i, st_map, False) for i, c in same_nb]
    intro = [_cell_view(c, chap.start + k, st_map, False) for k, c in enumerate(chap.cells[:3]) if c.get("cell_type") == "markdown"]
    expl = [_cell_view(nb.cells[i], i, st_map, False) for i in range(chap.start, chap.end)
        if nb.cells[i].get("cell_type") == "markdown" and any(abs(i - di) <= 2 for di in def_idxs) and not heading_info(nb.cells[i])]
    ex_local, ex_dist = [], []
    for idx, cell in enumerate(nb.cells):
        if idx in def_idxs or cell.get("cell_type") != "code" or is_exported(cell): continue
        if not _cell_mentions_symbol(cell, short_name): continue
        if chap.start <= idx < chap.end: ex_local.append(_cell_view(cell, idx, st_map, True))
        else: ex_dist.append((_chapter_for_idx(chaps, idx).title, str(cell.get("id", ""))))
    def_keys = {(str(nb_path.resolve()), d.cell_id) for d in defs}
    callers = _find_callers(short_name, scope_path if scope_path.exists() else nb_path.parent, def_keys)
    return SymbolContext(sym, _rel_label(nb_path, project_root(nb_path)), chap.title, intro, defs, expl, ex_local, ex_dist, callers)

Looking up a symbol bundles its local explanation, definition, nearby assertion, and cross-notebook caller into one view.

In [ ]:
with TemporaryDirectory() as tmp:
    root = Path(tmp)
    write_nb(new_nb([mk_cell("## Tax", "markdown"), mk_cell("`vat` computes 19% VAT.", "markdown"),
        mk_cell("#| export\ndef vat(x): return round(x * 0.19, 2)"), mk_cell("assert vat(100) == 19.0")]), root / "00_tax.ipynb")
    write_nb(new_nb([mk_cell("## Invoice", "markdown"), mk_cell("total = 100 + vat(100)")]), root / "01_inv.ipynb")
    ctx = symbol_context("vat", scope=root)
    test_eq((ctx.chapter, len(ctx.explanations), len(ctx.examples), len(ctx.callers)), ("Tax", 1, 1, 2))
    test_fail(lambda: symbol_context("vta", scope=root), contains="Closest: vat")
ctx

## Regex search across notebooks

When running standalone without `rgapi`, `search_nb` finds regex matches across notebook cells with chapter labels and cell IDs.

In [ ]:
#| export
@dataclass
class SearchHit:
    "One regex match inside a notebook cell."
    path: str
    chapter: str
    cell_id: str
    cell_type: str
    lineno: int
    preview: str

@dataclass
class SearchHits:
    "Collection of notebook search hits with compact display."
    pattern: str
    hits: list[SearchHit]

    def __len__(self) -> int: return len(self.hits)

    def __str__(self) -> str:
        if not self.hits: return f"No matches for {self.pattern!r}"
        return "\n".join(f"{h.path}#{h.cell_id}:{h.lineno} [{h.chapter}] ({h.cell_type}) {h.preview}" for h in self.hits)

    def _repr_markdown_(self) -> str: return f"```text\n{self}\n```"

`search_nb` uses smart-case matching (case-insensitive unless `pattern` includes uppercase letters) and supports filtering by cell type or chapter.

In [ ]:
#| export
def search_nb(
    pattern: str,  # Regex pattern (smart-case)
    path: str | Path = ".",  # Notebook or directory to search
    *,
    cell_type: str | None = None,  # Optional 'code' or 'markdown' filter
    chapter: str | None = None,  # Optional case-insensitive chapter substring filter
    limit: int = 30,  # Maximum hits to return
) -> SearchHits:
    "Search notebook cell sources with chapter context and stable cell IDs."
    flags = 0 if any(ch.isupper() for ch in pattern) else re.IGNORECASE
    rx = re.compile(pattern, flags)
    p = Path(path).expanduser()
    root = project_root(p)
    hits = []
    for nb_path in notebook_paths(p):
        nb = read_nb(nb_path)
        rel = _rel_label(nb_path, root)
        for ch in chapters(nb):
            if chapter and chapter.lower() not in ch.title.lower(): continue
            for cell in ch.cells:
                ct = cell.get("cell_type", "")
                if cell_type and ct != cell_type: continue
                cid = str(cell.get("id", ""))
                for lineno, line in enumerate(cell_source(cell).splitlines(), 1):
                    if rx.search(line):
                        hits.append(SearchHit(rel, ch.title, cid, ct, lineno, line.strip()[:140]))
                        if len(hits) >= limit: return SearchHits(pattern, hits)
    return SearchHits(pattern, hits)

Searching filters cleanly by chapter and cell type while returning exact line numbers and cell IDs.

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp) / "s.ipynb"
    write_nb(new_nb([mk_cell("## Alpha", "markdown"), mk_cell("needle = 1"), mk_cell("## Beta", "markdown"), mk_cell("needle = 2")]), p)
    res = search_nb("needle", p, chapter="beta", cell_type="code")
    test_eq((len(res), res.hits[0].chapter, res.hits[0].preview), (1, "Beta", "needle = 2"))
res